In [17]:
import joblib
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression 
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import (
                                        StratifiedKFold,
                                        cross_validate
                                    )
warnings.filterwarnings('ignore')

In [18]:
X_train = np.load("../../data/processed/X_train.npz")['arr_0']
Y_train = np.load("../../data/processed/Y_train.npz")['arr_0']
X_test = np.load("../../data/processed/X_test.npz")['arr_0']
Y_test = np.load("../../data/processed/Y_test.npz")['arr_0']

### 1. Define Parameter Grid

In [19]:
lr_param_grid = {
                "max_iter" : [1000, 5000, 10000]
                }

dt_param_grid = {
                "max_depth" : [8, 12, 16, 20],
                "criterion" : ["gini", "entrophy", "log_loss"]
                }

rf_param_grid = {
                "n_estimators" : [50, 100, 150, 200],
                "max_depth" : [8, 12, 16, 20],
                "criterion" : ["gini", "entrophy", "log_loss"]
                }

parameter_grid = {
                "Logistic Regression" : lr_param_grid,
                "Decision Tree" : dt_param_grid,
                "Random Forest" : rf_param_grid
                }

### 2. Define Model

In [20]:
models = {
            'Logistic Regression' : LogisticRegression(),
            'Decision Tree' : DecisionTreeClassifier(),
            'Random Forest' : RandomForestClassifier()
        }

### 3. Configure K-Folds

In [21]:
cv = StratifiedKFold(
                        n_splits=5,
                        shuffle=True,
                        random_state=42
                    )

### 4. Multimodel Training with Grid Search

In [23]:
from sklearn.model_selection import GridSearchCV


grid_search_result = {}

for model_name, model in models.items():
    print(f"{model_name} Tuning\n")
    param_grid = parameter_grid[model_name]
    grid_search = GridSearchCV(
                                estimator = model,
                                param_grid = param_grid,
                                cv = cv,
                                scoring = "f1",
                                verbose = 1,
                                return_train_score = False
                                )
    print(f"Filtering GridSearchCV for {model_name}")
    grid_search.fit(X_train, Y_train)
    grid_search_result[model_name] = grid_search

    print(f"{model_name} gridSearchCV completed")
    print(f"Best Parameters : {grid_search.best_params_}")
    print(f"Best CV score : {grid_search.best_score_}")


Logistic Regression Tuning

Filtering GridSearchCV for Logistic Regression
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Logistic Regression gridSearchCV completed
Best Parameters : {'max_iter': 5000}
Best CV score : 0.7821845741596369
Decision Tree Tuning

Filtering GridSearchCV for Decision Tree
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Decision Tree gridSearchCV completed
Best Parameters : {'criterion': 'log_loss', 'max_depth': 8}
Best CV score : 0.8418877967524849
Random Forest Tuning

Filtering GridSearchCV for Random Forest
Fitting 5 folds for each of 48 candidates, totalling 240 fits
Random Forest gridSearchCV completed
Best Parameters : {'criterion': 'log_loss', 'max_depth': 8, 'n_estimators': 200}
Best CV score : 0.8609151305331721
